# Session 8 · How Good Is My Model? Regression Metrics

**Machine Learning Foundations · Sanketana School of Code**

For three sessions we drove the **cost** down — 44, then 26 — but never answered the obvious question: **is 26 good?** And "26" is no use to a parent: it's in *points squared*, measured on data the model already saw. Today we fix both, and turn cost into numbers a human can actually use.

By the end of this notebook you will be able to:

- explain why we judge a model on a **held-out test set**
- compute **RMSE** (= √cost), **MAE**, and **R²**, and say what each means in plain words
- compare a model to a **guess-the-average baseline** and decide if it's good *for this problem*

**New rule from today on: _numbers, not vibes._** Every model you build is reported with a named metric.

## Warm-up · Last session's homework

Your coach will walk through Session 7's housing coefficients (about 10 minutes). You ranked the features with the *scaled* table.

And you were left with the question that closes this module: the cost dropped nicely — but *how good is the model, in a number you could actually report?*

## Step 1 · Two things wrong with "cost = 26"

1. **Wrong units.** Cost is the average *squared* miss — "points squared." Nobody thinks in points squared.
2. **Measured on seen data.** We computed it on the rows the model trained on — the memorising trap from Sessions 1 and 4.

Two fixes: put the error back in **plain points**, and measure it on a **held-out test set**.

## Step 2 · Bring back the split

To judge honestly, score on data the model never trained on. The `train_test_split` from Session 4 returns — fit on train, measure on test.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

students = pd.read_csv("../../../datasets/anchor/student_habits.csv")
habits = ["study_hours_per_week", "attendance_pct", "sleep_hours_per_night",
          "screen_time_hours_per_day", "practice_sessions_per_week"]
X = students[habits]
y = students["test_score"]

# ✏️ TODO: hide 20% as a test set, then fit on the training set only.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
model = LinearRegression().fit(X_train, y_train)

print("training rows:", len(X_train), " | hidden test rows:", len(X_test))

# The model's predictions on the unseen test students:
pred = model.predict(X_test)

## Step 3 · RMSE = √cost, back in plain points

The nicest idea of the day. Take the **cost** (average squared miss) and take its **square root**. Squaring pushed cost into "points squared"; the root pulls it back to plain points. That's **RMSE**.

In [ ]:
from sklearn.metrics import mean_squared_error

cost = mean_squared_error(y_test, pred)   # average squared miss — Session 6's cost
rmse = np.sqrt(cost)

print(f"test cost (average squared miss): {cost:.1f}   ← points squared, not human")
print(f"RMSE = square root of the cost:   {rmse:.2f}   ← plain points!")
print()
print(f"So: the model's predictions are typically off by about {rmse:.0f} points.")

RMSE is just Session 6's cost wearing human clothes. **"Off by about 5 points"** is a sentence anyone understands.

## Step 4 · MAE — the plainest error of all

**MAE** (mean absolute error) is even simpler: the average size of the miss, ignoring sign.

In [ ]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, pred)
print(f"MAE:  {mae:.2f} points   (average size of a miss)")
print(f"RMSE: {rmse:.2f} points   (same idea, but big misses count more)")

print("\nRMSE is always ≥ MAE: squaring makes a few big misses hurt more.")
print("Use MAE for the typical error; RMSE when large mistakes are especially bad.")

## Step 5 · R² — how much better than just guessing?

Now the mystery number. Back in Session 4, `.score()` printed something like `0.89` and we said "higher is better." That number is **R²**. Here's what it means: compare your model to the **laziest possible model** — one that ignores every feature and just predicts the **average** score for everyone.

In [ ]:
from sklearn.metrics import r2_score

# The lazy baseline: predict the average TRAINING score for every test student.
baseline = np.full(len(y_test), y_train.mean())

print(f"guess-the-average baseline:  RMSE {np.sqrt(mean_squared_error(y_test, baseline)):5.1f}   R2 {r2_score(y_test, baseline):+.2f}")
print(f"our five-habit model:        RMSE {rmse:5.1f}   R2 {r2_score(y_test, pred):+.2f}")

# .score() returns exactly R2 — the Session 4 mystery, solved:
print(f"\nmodel.score(X_test, y_test) = {model.score(X_test, y_test):.3f}  (this IS R2)")

Read it: the baseline scores **R² ≈ 0** (no better than guessing the average). Our model scores **≈ 0.89** — it explains about **89% of the variation** in scores the average alone couldn't.

> **Word carefully:** R² is "89% of the *variation* explained," **not** "89% of predictions correct." And R² can go *below 0* — that's a model worse than guessing the average.

## Step 6 · Is it good? Depends on the problem

A metric is never good in a vacuum. Judge it against the **label's scale** and the **baseline**.

In [ ]:
print(f"test scores range from {y.min()} to {y.max()}")
print(f"guessing the average is off by ~{np.sqrt(mean_squared_error(y_test, baseline)):.0f} points")
print(f"our model is off by only     ~{rmse:.0f} points  → about {np.sqrt(mean_squared_error(y_test, baseline))/rmse:.0f}× better")

### ✏️ Say the verdict — with numbers

Finish the sentence using the numbers above: "Our model predicts test scores to within about ____ points, and explains about ____% of the variation — roughly ____× better than guessing the average."

*Your verdict:* 

## Step 7 · Train vs test — did it learn or memorise?

The Session 4 check, now with real numbers. Compare the metrics on data the model saw (train) and didn't (test).

In [ ]:
for name, Xs, ys in [("TRAIN (seen)   ", X_train, y_train), ("TEST  (unseen) ", X_test, y_test)]:
    p = model.predict(Xs)
    print(f"{name}  MAE {mean_absolute_error(ys, p):.2f}   RMSE {np.sqrt(mean_squared_error(ys, p)):.2f}   R2 {r2_score(ys, p):.3f}")

print("\nTrain and test are close → the model LEARNED, it didn't memorise.")
print("A big train-over-test gap would be the warning sign we study in Session 16.")

### ✏️ Reflect

1. Are the train and test numbers close? What would a *large* gap have told you?
2. Why do we always report the **test** metric, never the training one?

*Your answers:*

1. 
2. 

## What we learned

✏️ Three quick reflections — one line each:

1. RMSE is the square root of ____, which puts the error back in ____.
2. R² = 0 means the model is no better than ____.
3. "Good" for a metric depends on ____.

---

**You can now report a model like a professional** — off by about 5 points, R² ≈ 0.89, far better than guessing. That closes **Module 2**: you walked *data → model → evaluation → insight* with **evaluation** finally done in named numbers. And the course rule is now live: **numbers, not vibes** — every model from here on carries a named metric.

**Next (Module 3): classification.** The target stops being a number and becomes a *category* — spam or not, pass or fail. RMSE and R² won't fit that job, so we'll meet a new family of metrics built for categories. Same workflow, new kind of question.